## Disclaimer

**Environment:** This notebook has been tested with:

- Julia v1.12.x
- JuMP.jl v1.30.x
- HiGHS v1.24.1

**Author:** Xavier Gandibleux, Nantes (France)


----

## Case 1: Single-Objective Optimization

Given the 01 Unidimensional Knapsack Problem (01UKP) formulated by 
$$z=\max\big\{px \mid wx \le c, \ x\in\{0,1\}^n\big\}$$

and the numerical instance corresponding to 
$$n=5$$
$$p=(5, 3, 2, 7, 4)$$
$$w=(2, 8, 4, 2, 5)$$
$$c=10$$

perform the following progressive tasks:

1. implement an explicit formulation of the 01UKP with JuMP and display the model.
2. compute the optimal solution using HiGHS as MIP solver and display the solution.
3. compute the linear relaxation using HiGHS as LP solver and display the solution.
4. add an constraint and examine the optimal solution from different perspectives.


## Solutions:

----
### 1. implement and display an explicit formulation of the 01UKP with JuMP.

<ins>indications:</ins> an explicit formulation is a description in extension of a mathematical model (composed of variable(s), function(s), constraint(s)), and data. 
<br>

<ins>Hints:</ins> the formulation to implement in JuMP is the following:
$$ \begin{aligned}
\max\quad & 5 x_1 + 3 x_2 + 2 x_3 + 7 x_4 + 4 x_5\\
\text{Subject to} \quad & 2 x_1 + 8 x_2 + 4 x_3 + 2 x_4 + 5 x_5 \leq 10\\
 & x_1, x_2, x_3, x_4, x_5 \in \{0, 1\}\\
\end{aligned} $$

#### declare the package(s) to use:

In [ ]:
using JuMP, HiGHS

<br>

#### setup the formulation:

In [ ]:
ukp = Model( )

@variable(ukp, x1, Bin)
@variable(ukp, x2, Bin)
@variable(ukp, x3, Bin)
@variable(ukp, x4, Bin)
@variable(ukp, x5, Bin)
@objective(ukp, Max, 5x1 + 3x2 + 2x3 + 7x4 + 4x5)
@constraint(ukp, 2x1 + 8x2 + 4x3 + 2x4 + 5x5 ≤ 10)

<br>

#### display the model:
<ins>Hints:</ins> see `print()` in the Julia documentation

In [ ]:
print(ukp)

<br>

----
### 2. Compute and display the optimal solution of the 01UKP solved using HiGHS as MIP solver. 

#### setup the MIP solver to use:

In [ ]:
set_optimizer(ukp, HiGHS.Optimizer)
set_silent(ukp)

<br>

#### compute the optimal solution:

In [ ]:
optimize!(ukp)

<br>

#### display the results:

In [ ]:
if is_solved_and_feasible(ukp)
    println("zOpt: ", objective_value(ukp))
    println("xOpt: ", value(x1),"|1  ", value(x2),"|2  ", value(x3),"|3  ", value(x4),"|4  ", value(x5),"|5")
end

<br>
<ins> Technical note:</ins> use ` value( ukp[ :x1 ] ) ` for refering a variable using the name of a model

<br>
<ins> Remark:</ins> in general, an MIP solver handles all decision variables as floating-point variables. The returned values may be approximations of integer values. Improve the results accordingly by displaying the integer value. 

<ins> Hints:</ins> see functions `convert()` and `round()` in the Julia documentation.

In [ ]:
convert(Int,value(x1))

In [ ]:
round(0.999,digits=0)

In [ ]:
convert(Int, round(0.999,digits=0))

In [ ]:
if is_solved_and_feasible(ukp)
    println("zOpt: ", convert(Int, round(objective_value(ukp))))
    println("xOpt: ", convert(Int, round(value(x1),digits=0)),"|1  ",
                      convert(Int, round(value(x2),digits=0)),"|2  ", 
                      convert(Int, round(value(x3),digits=0)),"|3  ", 
                      convert(Int, round(value(x4),digits=0)),"|4  ", 
                      convert(Int, round(value(x5),digits=0)),"|5")
end

<br> 

----
### 3. Compute the linear relaxation of the UKP using HiGHS as LP solver and display the solution.
<ins>Hints:</ins> see the type of variables of a JuMP model

In [ ]:
kpLP = Model( )

@variable(kpLP, 0 ≤ x1lp ≤ 1)
@variable(kpLP, 0 ≤ x2lp ≤ 1)
@variable(kpLP, 0 ≤ x3lp ≤ 1)
@variable(kpLP, 0 ≤ x4lp ≤ 1)
@variable(kpLP, 0 ≤ x5lp ≤ 1)
@objective(kpLP, Max, 5x1lp + 3x2lp + 2x3lp + 7x4lp + 4x5lp)
@constraint(kpLP, 2x1lp + 8x2lp + 4x3lp + 2x4lp + 5x5lp ≤ 10)

set_optimizer(kpLP, HiGHS.Optimizer)
set_silent(kpLP)

optimize!(kpLP)

if is_solved_and_feasible(kpLP)
    println("zOpt: ", objective_value(kpLP))
    println("xOpt: ", value(x1lp),"|1  ", value(x2lp),"|2  ", value(x3lp),"|3  ", value(x4lp),"|4  ", value(x5lp),"|5")
end

<ins>Hints:</ins> see `relax_integrality()` in the JuMP documentation

In [ ]:
relax = relax_integrality(ukp) # all variables are continuous and 0 ≤ x_i ≤ 1
print(ukp)

optimize!(ukp)
println("zOpt: ", objective_value(ukp))
println("xOpt:  1| ", value(x1),"  2| ", value(x2),"  3| ", value(x3),"  4| ", value(x4),"  5| ", value(x5))

relax() # restore the definition of variables

<br> 

----
### 4. Add an constraint and examine the optimal solution from different perspectives

Let's transform the optimization problem into a two-dimensional knapsack problem (01BKP) by adding the following constraint:

$$6x_1​+2x_2​+5x_3​+x4_​+3x_5 ​\le 10$$

<br>

##### 1. Modify the model and solve it
**Tasks:** add this constraint to JuMP model and solve it with HiGHS. Print the optimal objective value and the selected variables.

<ins>Indication:</ins> let `p`, `w1`and `w2` be the vectors of coefficients for the profits and weights both both constraints. Write the model in implicit form.

<ins>Hints:</ins> See the definition of a vector in the Julia documentation

<ins>Suggestion:</ins> name each constraint `cst1` and `cst2`, respectively.

In [ ]:
p  = [5, 3, 2, 7, 4]
w1 = [2, 8, 4, 2, 5]
w2 = [6, 2, 5, 1, 3]
n = length(p)

bkp = Model(HiGHS.Optimizer)
@variable(bkp, x[1:n], Bin)
@objective(bkp, Max, sum(p[i]*x[i] for i in 1:n))
@constraint(bkp, cst1, sum(w1[i]*x[i] for i in 1:n) <= 10)
@constraint(bkp, cst2, sum(w2[i]*x[i] for i in 1:n) <= 10)

set_silent(bkp)
optimize!(bkp)

println("Optimal value: ", objective_value(bkp))
println("Selected items: ", value.(x))

<br>

##### 2. Reading the solution

**Task**: Which items are selected? What is the remaining slack on each constraint?  

<ins> Hints:</ins> see instruction `for` and `if`; see function `sum`

In [ ]:
selected = [i for i in 1:n if value(x[i]) > 0.5]
println("Selected items: ", selected)

slack1 = 10 - sum(w1[i]*value(x[i]) for i in 1:n)
slack2 = 10 - sum(w2[i]*value(x[i]) for i in 1:n)
println("Slack constraint 1: ", slack1)
println("Slack constraint 2: ", slack2)

<br>

##### 3. Simple sensitivity analysis

**Task**: If capacity of constraint 1 increases from 10 to 12, does the solution change?

<ins> Hints:</ins> see instruction `set_normalized_rhs`

In [ ]:
println("Capacity 10 -> obj = ", objective_value(bkp))

# Modify only the RHS of constraint cst1, no need to rebuild the model
set_normalized_rhs(cst1, 12)
print(bkp)

optimize!(bkp)
println("Capacity 12 -> obj = ", objective_value(bkp))

Comments :

- Constraints must be named (e.g. `cst1`, `cst2`) when declared with `@constraint(model, cst1, ...)` so they can be referenced afterward.
- `set_normalized_rhs(constraint_ref, new_value)` changes the RHS in place. The model structure, variables, and objective stay untouched.

This is much faster than reconstructing the model, especially useful for parametric studies (e.g., looping over several capacity values).